In [1]:
import os
import torch
import torchvision
from torchvision import transforms,datasets
from torch.utils.data import DataLoader,random_split
import torch.nn as nn
import torch.optim as optim

In [2]:
print(torch.__version__)

2.5.1+cu121


In [7]:
transform= transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [9]:
dataset_path=r"C:\Users\vpokh\Downloads\archive (3)\new_violence"

dataset= datasets.ImageFolder(root=dataset_path,transform=transform)
print("Classes:", dataset.classes)
print("Total images:", len(dataset))

Classes: ['non_violence', 'violence']
Total images: 11073


In [10]:
train_size=int(0.8 * len(dataset))
test_size=len(dataset)-train_size
train_data , test_data= random_split(dataset,[train_size,test_size])
print("Train:", len(train_data))
print("Test:", len(test_data))


Train: 8858
Test: 2215


In [11]:
train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
test_loader=DataLoader(test_data,batch_size=32,shuffle=False)

In [12]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32])


In [13]:
model= torchvision.models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad=False

model.fc = nn.Linear(model.fc.in_features,2)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

c:\Users\vpokh\anaconda3\envs\yolo_cuda\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\vpokh\anaconda3\envs\yolo_cuda\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer= optim.Adam(model.fc.parameters(),lr=0.001)

In [16]:
epochs=3

for epoch in range(epochs):
    model.train()
    total_loss=0

    for images,labels in train_loader:
        images,labels = images.to(device),labels.to(device)

        outputs=model(images)
        loss=criterion(outputs,labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss +=loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 114.7258
Epoch 2, Loss: 79.0782
Epoch 3, Loss: 75.2671


In [17]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 91.02%


In [19]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": dataset.classes
}, "fight_model.pth")